In [1]:
pip install mysql-connector-python pandas pytest

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/17.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/17.7 MB ? eta -:--:--
    --------------------------------------- 0.3/17.7 MB ? eta -:--:--
   - -------------------------------------- 0.5/17.7 MB 1.4 MB/s eta 0:00:13
   - -------------------------------------- 0.8/17.7 MB 1.4 MB/s eta 0:00:13
   -- ------------------------------------- 1.3/17.7 MB 1.4 MB/s eta 0:00:12
   --- ------------------------------------ 1.6/17.7 MB 1.5 MB/s eta 0:00:12
   --- ------------------------------------ 1.6/17.7 MB 1.5 MB/s eta 0:00:12
   --- ------------------------------------ 1.6/17.7 MB 1.5 MB/s eta 0:00:12
   ---- ----------------------------------- 1.8/17.7 MB 977.5 kB/s eta 0:00:17
   ---- ----------------------------------- 2.1/17.7 MB 1.1 MB/s eta 0:00:15
   ----- ---------------------------------- 2.6/17.7 MB 1.2 MB/s eta 0:00:13
   ------ -------

In [1]:
import mysql.connector
from getpass import getpass

password = getpass("Enter your MySQL password: ")

connection = mysql.connector.connect(
    host="localhost",
    user="root",
    password=password,
    database="yottaflex_pipeline"
)

if connection.is_connected():
    print("Successfully connected to YottaFlex database!")

connection.close()

Enter your MySQL password:  ········


Successfully connected to YottaFlex database!


In [ ]:
import mysql.connector
from getpass import getpass

DB_CONFIG = {
    "host": "localhost",
    "user": "root",
    "password": getpass("Enter your MySQL password: "),
    "database": "yottaflex_pipeline"
}

def get_connection():
    return mysql.connector.connect(**DB_CONFIG)

In [3]:
def start_pipeline_run():
    connection = get_connection()
    cursor = connection.cursor()

    start_time = datetime.now(timezone.utc).replace(tzinfo=None)

    query = """
        INSERT INTO pipeline_runs (start_time, status)
        VALUES (%s, %s)
    """

    cursor.execute(query, (start_time, "RUNNING"))

    run_id = cursor.lastrowid

    connection.commit()
    cursor.close()
    connection.close()

    return run_id

In [4]:
run_id = start_pipeline_run()

print("Pipeline started!")
print("Run ID:", run_id)

Pipeline started!
Run ID: 1


In [5]:
from datetime import datetime

def validate_event(event):

    errors = []

    # Check required fields
    required_fields = [
        "event_id",
        "customer_id",
        "event_type",
        "event_version",
        "event_timestamp"
    ]

    for field in required_fields:
        if event.get(field) is None or str(event.get(field)).strip() == "":
            errors.append(f"Missing required field: {field}")

    # Stop further checks if essential fields are missing
    if errors:
        return False, errors

    # Check event type
    event_type = str(event["event_type"]).upper()

    if event_type not in ["INSERT", "UPDATE", "DELETE"]:
        errors.append("Invalid event type")

    # Check event version
    version = event["event_version"]

    if isinstance(version, bool) or not isinstance(version, int) or version <= 0:
        errors.append("Event version must be a positive integer")

    # Check timestamp
    try:
        datetime.fromisoformat(
            event["event_timestamp"].replace("Z", "+00:00")
        )
    except (ValueError, TypeError, AttributeError):
        errors.append("Invalid event timestamp")

    # Check customer details for INSERT and UPDATE
    if event_type in ["INSERT", "UPDATE"]:

        if not event.get("customer_name"):
            errors.append("Customer name is required")

        if not event.get("email"):
            errors.append("Email is required")

        elif "@" not in event["email"]:
            errors.append("Invalid email format")

    return len(errors) == 0, errors

In [6]:
valid_event = {
    "event_id": "E001",
    "customer_id": "C101",
    "event_type": "INSERT",
    "event_version": 1,
    "event_timestamp": "2026-11-13T10:00:00Z",
    "customer_name": "Ravi",
    "email": "ravi@example.com",
    "source_file": "customers_batch_01.json"
}

is_valid, errors = validate_event(valid_event)

print("Valid:", is_valid)
print("Errors:", errors)

Valid: True
Errors: []


In [7]:
invalid_event = {
    "event_id": "E002",
    "customer_id": "C102",
    "event_type": "INSERT",
    "event_version": 0,
    "event_timestamp": "wrong-date",
    "customer_name": "",
    "email": "invalid-email",
    "source_file": "customers_batch_01.json"
}

is_valid, errors = validate_event(invalid_event)

print("Valid:", is_valid)
print("Errors:", errors)

Valid: False
Errors: ['Event version must be a positive integer', 'Invalid event timestamp', 'Customer name is required', 'Invalid email format']


In [8]:
def is_duplicate_event(event_id):

    connection = get_connection()
    cursor = connection.cursor()

    query = """
        SELECT COUNT(*)
        FROM customer_events
        WHERE event_id = %s
    """

    cursor.execute(query, (event_id,))

    count = cursor.fetchone()[0]

    cursor.close()
    connection.close()

    return count > 0

In [11]:
print("E001 duplicate:", is_duplicate_event("E001"))
print("E999 duplicate:", is_duplicate_event("E999"))

E001 duplicate: False
E999 duplicate: False


In [12]:
def insert_customer_event(event, run_id):

    is_valid, errors = validate_event(event)

    if not is_valid:
        print("Event rejected:", errors)
        return False

    connection = get_connection()
    cursor = connection.cursor()

    try:
        query = """
            INSERT INTO customer_events (
                event_id,
                customer_id,
                event_type,
                event_version,
                event_timestamp,
                customer_name,
                email,
                source_file,
                run_id
            )
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
        """

        event_timestamp = datetime.fromisoformat(
            event["event_timestamp"].replace("Z", "+00:00")
        ).replace(tzinfo=None)

        values = (
            event["event_id"],
            event["customer_id"],
            event["event_type"].upper(),
            event["event_version"],
            event_timestamp,
            event.get("customer_name"),
            event.get("email"),
            event.get("source_file"),
            run_id
        )

        cursor.execute(query, values)
        connection.commit()

        print("Event inserted successfully!")
        return True

    except mysql.connector.IntegrityError:
        connection.rollback()
        print("Duplicate or conflicting event detected!")
        return False

    except Exception:
        connection.rollback()
        raise

    finally:
        cursor.close()
        connection.close()

In [13]:
insert_customer_event(valid_event, run_id)

Event inserted successfully!


True

In [14]:
print(is_duplicate_event("E001"))

True


In [15]:
def update_current_customer(event):

    connection = get_connection()
    cursor = connection.cursor()

    try:
        connection.start_transaction()

        customer_id = event["customer_id"]
        event_type = event["event_type"].upper()
        version = event["event_version"]

        # Check whether the customer already exists
        cursor.execute(
            """
            SELECT latest_version
            FROM current_customers
            WHERE customer_id = %s
            FOR UPDATE
            """,
            (customer_id,)
        )

        existing = cursor.fetchone()

        # First event for this customer
        if existing is None:

            if event_type != "INSERT":
                raise ValueError(
                    "Customer does not exist. INSERT event required first."
                )

            cursor.execute(
                """
                INSERT INTO current_customers (
                    customer_id,
                    customer_name,
                    email,
                    latest_version,
                    last_event_id,
                    last_event_timestamp,
                    is_deleted
                )
                VALUES (%s, %s, %s, %s, %s, %s, FALSE)
                """,
                (
                    customer_id,
                    event["customer_name"],
                    event["email"],
                    version,
                    event["event_id"],
                    datetime.fromisoformat(
                        event["event_timestamp"].replace("Z", "+00:00")
                    ).replace(tzinfo=None)
                )
            )

            print("New customer inserted into current state.")

        # Customer already exists
        elif version > existing[0]:

            if event_type == "DELETE":
                cursor.execute(
                    """
                    UPDATE current_customers
                    SET latest_version = %s,
                        last_event_id = %s,
                        last_event_timestamp = %s,
                        is_deleted = TRUE
                    WHERE customer_id = %s
                    """,
                    (
                        version,
                        event["event_id"],
                        datetime.fromisoformat(
                            event["event_timestamp"].replace("Z", "+00:00")
                        ).replace(tzinfo=None),
                        customer_id
                    )
                )

                print("Customer marked as deleted.")

            else:
                cursor.execute(
                    """
                    UPDATE current_customers
                    SET customer_name = %s,
                        email = %s,
                        latest_version = %s,
                        last_event_id = %s,
                        last_event_timestamp = %s,
                        is_deleted = FALSE
                    WHERE customer_id = %s
                    """,
                    (
                        event["customer_name"],
                        event["email"],
                        version,
                        event["event_id"],
                        datetime.fromisoformat(
                            event["event_timestamp"].replace("Z", "+00:00")
                        ).replace(tzinfo=None),
                        customer_id
                    )
                )

                print("Customer state updated.")

        else:
            print("Stale event: history retained, current state unchanged.")

        connection.commit()

    except Exception:
        connection.rollback()
        raise

    finally:
        cursor.close()
        connection.close()

In [16]:
update_current_customer(valid_event)

New customer inserted into current state.


In [17]:
update_event = {
    "event_id": "E002",
    "customer_id": "C101",
    "event_type": "UPDATE",
    "event_version": 2,
    "event_timestamp": "2026-11-13T11:00:00Z",
    "customer_name": "Ravi Kumar",
    "email": "ravikumar@example.com",
    "source_file": "customers_batch_01.json"
}

In [18]:
if not is_duplicate_event(update_event["event_id"]):
    if insert_customer_event(update_event, run_id):
        update_current_customer(update_event)

Event inserted successfully!
Customer state updated.


In [19]:
late_event = {
    "event_id": "E003",
    "customer_id": "C101",
    "event_type": "UPDATE",
    "event_version": 1,
    "event_timestamp": "2026-11-13T09:00:00Z",
    "customer_name": "Ravi Old",
    "email": "old@example.com",
    "source_file": "late_batch.json"
}

In [20]:
if not is_duplicate_event(late_event["event_id"]):
    if insert_customer_event(late_event, run_id):
        update_current_customer(late_event)

Duplicate or conflicting event detected!


In [21]:
customer_b_insert = {
    "event_id": "E010",
    "customer_id": "C102",
    "event_type": "INSERT",
    "event_version": 3,
    "event_timestamp": "2026-11-13T12:00:00Z",
    "customer_name": "Arun",
    "email": "arun@example.com",
    "source_file": "customers_batch_02.json"
}

In [22]:
if not is_duplicate_event(customer_b_insert["event_id"]):
    if insert_customer_event(customer_b_insert, run_id):
        update_current_customer(customer_b_insert)

Event inserted successfully!
New customer inserted into current state.


In [23]:
customer_b_old = {
    "event_id": "E011",
    "customer_id": "C102",
    "event_type": "UPDATE",
    "event_version": 2,
    "event_timestamp": "2026-11-13T11:00:00Z",
    "customer_name": "Arun Old",
    "email": "old@example.com",
    "source_file": "late_batch.json"
}

In [24]:
if not is_duplicate_event(customer_b_old["event_id"]):
    if insert_customer_event(customer_b_old, run_id):
        update_current_customer(customer_b_old)

Event inserted successfully!
Stale event: history retained, current state unchanged.


In [25]:
delete_event = {
    "event_id": "E004",
    "customer_id": "C101",
    "event_type": "DELETE",
    "event_version": 3,
    "event_timestamp": "2026-11-13T13:00:00Z",
    "customer_name": None,
    "email": None,
    "source_file": "customers_batch_01.json"
}

In [26]:
if not is_duplicate_event(delete_event["event_id"]):
    if insert_customer_event(delete_event, run_id):
        update_current_customer(delete_event)

Event inserted successfully!
Customer marked as deleted.


In [27]:
import json

def reject_event(event, errors, run_id):

    connection = get_connection()
    cursor = connection.cursor()

    query = """
        INSERT INTO rejected_events (
            event_id,
            customer_id,
            raw_event,
            rejection_reason,
            source_file,
            run_id
        )
        VALUES (%s, %s, %s, %s, %s, %s)
    """

    values = (
        event.get("event_id"),
        event.get("customer_id"),
        json.dumps(event, default=str),
        "; ".join(errors),
        event.get("source_file"),
        run_id
    )

    try:
        cursor.execute(query, values)
        connection.commit()
        print("Rejected event stored successfully!")

    finally:
        cursor.close()
        connection.close()

In [28]:
bad_event = {
    "event_id": "E100",
    "customer_id": "C105",
    "event_type": "INSERT",
    "event_version": 0,
    "event_timestamp": "wrong-date",
    "customer_name": "",
    "email": "invalid-email",
    "source_file": "bad_batch.json"
}

In [29]:
is_valid, errors = validate_event(bad_event)

if not is_valid:
    reject_event(bad_event, errors, run_id)

Rejected event stored successfully!


In [30]:
def finish_pipeline_run(run_id):

    connection = get_connection()
    cursor = connection.cursor()

    try:
        # Count accepted events
        cursor.execute("""
            SELECT COUNT(*)
            FROM customer_events
            WHERE run_id = %s
        """, (run_id,))

        accepted_count = cursor.fetchone()[0]

        # Count rejected events
        cursor.execute("""
            SELECT COUNT(*)
            FROM rejected_events
            WHERE run_id = %s
        """, (run_id,))

        rejected_count = cursor.fetchone()[0]

        # Count accepted events that did not update current state
        cursor.execute("""
            SELECT COUNT(*)
            FROM customer_events e
            JOIN current_customers c
                ON e.customer_id = c.customer_id
            WHERE e.run_id = %s
              AND e.event_version < c.latest_version
        """, (run_id,))

        stale_count = cursor.fetchone()[0]

        input_count = accepted_count + rejected_count

        cursor.execute("""
            UPDATE pipeline_runs
            SET end_time = %s,
                status = 'COMPLETED',
                input_count = %s,
                accepted_count = %s,
                rejected_count = %s,
                stale_count = %s
            WHERE run_id = %s
        """, (
            datetime.now(timezone.utc).replace(tzinfo=None),
            input_count,
            accepted_count,
            rejected_count,
            stale_count,
            run_id
        ))

        connection.commit()

        print("Pipeline run completed!")
        print("Accepted:", accepted_count)
        print("Rejected:", rejected_count)
        print("Stale:", stale_count)

    except Exception:
        connection.rollback()
        raise

    finally:
        cursor.close()
        connection.close()

In [31]:
finish_pipeline_run(run_id)

Pipeline run completed!
Accepted: 5
Rejected: 1
Stale: 3


In [32]:
import json

def process_event(event, run_id):

    connection = get_connection()
    cursor = connection.cursor()

    try:
        connection.start_transaction()

        # Count every incoming event attempt
        cursor.execute("""
            UPDATE pipeline_runs
            SET input_count = input_count + 1
            WHERE run_id = %s
        """, (run_id,))

        # 1. Check duplicate event ID
        cursor.execute("""
            SELECT event_id
            FROM customer_events
            WHERE event_id = %s
        """, (event.get("event_id"),))

        if cursor.fetchone():
            cursor.execute("""
                UPDATE pipeline_runs
                SET duplicate_count = duplicate_count + 1
                WHERE run_id = %s
            """, (run_id,))

            connection.commit()
            return "DUPLICATE"

        # 2. Validate event
        is_valid, errors = validate_event(event)

        if not is_valid:
            cursor.execute("""
                INSERT INTO rejected_events (
                    event_id, customer_id, raw_event,
                    rejection_reason, source_file, run_id
                )
                VALUES (%s, %s, %s, %s, %s, %s)
            """, (
                event.get("event_id"),
                event.get("customer_id"),
                json.dumps(event, default=str),
                "; ".join(errors),
                event.get("source_file"),
                run_id
            ))

            cursor.execute("""
                UPDATE pipeline_runs
                SET rejected_count = rejected_count + 1
                WHERE run_id = %s
            """, (run_id,))

            connection.commit()
            return "REJECTED"

        event_type = event["event_type"].upper()
        timestamp = datetime.fromisoformat(
            event["event_timestamp"].replace("Z", "+00:00")
        ).replace(tzinfo=None)

        # 3. Check the current customer version
        cursor.execute("""
            SELECT latest_version
            FROM current_customers
            WHERE customer_id = %s
            FOR UPDATE
        """, (event["customer_id"],))

        existing = cursor.fetchone()

        # An UPDATE or DELETE cannot create a new customer
        if existing is None and event_type != "INSERT":
            errors = ["Customer does not exist; INSERT required first."]

            cursor.execute("""
                INSERT INTO rejected_events (
                    event_id, customer_id, raw_event,
                    rejection_reason, source_file, run_id
                )
                VALUES (%s, %s, %s, %s, %s, %s)
            """, (
                event["event_id"],
                event["customer_id"],
                json.dumps(event, default=str),
                "; ".join(errors),
                event.get("source_file"),
                run_id
            ))

            cursor.execute("""
                UPDATE pipeline_runs
                SET rejected_count = rejected_count + 1
                WHERE run_id = %s
            """, (run_id,))

            connection.commit()
            return "REJECTED"

        # 4. Insert the accepted event into audit history
        cursor.execute("""
            INSERT INTO customer_events (
                event_id, customer_id, event_type,
                event_version, event_timestamp,
                customer_name, email, source_file, run_id
            )
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
        """, (
            event["event_id"],
            event["customer_id"],
            event_type,
            event["event_version"],
            timestamp,
            event.get("customer_name"),
            event.get("email"),
            event.get("source_file"),
            run_id
        ))

        # 5. Update current state only for a newer event
        if existing is None:

            cursor.execute("""
                INSERT INTO current_customers (
                    customer_id, customer_name, email,
                    latest_version, last_event_id,
                    last_event_timestamp, is_deleted
                )
                VALUES (%s, %s, %s, %s, %s, %s, FALSE)
            """, (
                event["customer_id"],
                event["customer_name"],
                event["email"],
                event["event_version"],
                event["event_id"],
                timestamp
            ))

        elif event["event_version"] > existing[0]:

            if event_type == "DELETE":
                cursor.execute("""
                    UPDATE current_customers
                    SET latest_version = %s,
                        last_event_id = %s,
                        last_event_timestamp = %s,
                        is_deleted = TRUE
                    WHERE customer_id = %s
                """, (
                    event["event_version"],
                    event["event_id"],
                    timestamp,
                    event["customer_id"]
                ))

            else:
                cursor.execute("""
                    UPDATE current_customers
                    SET customer_name = %s,
                        email = %s,
                        latest_version = %s,
                        last_event_id = %s,
                        last_event_timestamp = %s,
                        is_deleted = FALSE
                    WHERE customer_id = %s
                """, (
                    event["customer_name"],
                    event["email"],
                    event["event_version"],
                    event["event_id"],
                    timestamp,
                    event["customer_id"]
                ))

        else:
            # Older unique events remain in history
            cursor.execute("""
                UPDATE pipeline_runs
                SET stale_count = stale_count + 1
                WHERE run_id = %s
            """, (run_id,))

        cursor.execute("""
            UPDATE pipeline_runs
            SET accepted_count = accepted_count + 1
            WHERE run_id = %s
        """, (run_id,))

        connection.commit()
        return "ACCEPTED"

    except mysql.connector.IntegrityError:
        connection.rollback()
        raise

    except Exception:
        connection.rollback()
        raise

    finally:
        cursor.close()
        connection.close()

In [33]:
test_event = {
    "event_id": "E200",
    "customer_id": "C200",
    "event_type": "INSERT",
    "event_version": 1,
    "event_timestamp": "2026-11-13T14:00:00Z",
    "customer_name": "Kiran",
    "email": "kiran@example.com",
    "source_file": "integrated_test.json"
}

print(process_event(test_event, run_id))

ACCEPTED


In [34]:
print(process_event(test_event, run_id))

DUPLICATE


In [35]:
run_id_2 = start_pipeline_run()
print("New Run ID:", run_id_2)

New Run ID: 2


In [36]:
update_test = {
    "event_id": "E201",
    "customer_id": "C200",
    "event_type": "UPDATE",
    "event_version": 2,
    "event_timestamp": "2026-11-13T14:10:00Z",
    "customer_name": "Kiran Kumar",
    "email": "kirankumar@example.com",
    "source_file": "test_batch.json"
}

print(process_event(update_test, run_id_2))

ACCEPTED


In [37]:
delete_test = {
    "event_id": "E202",
    "customer_id": "C200",
    "event_type": "DELETE",
    "event_version": 3,
    "event_timestamp": "2026-11-13T14:20:00Z",
    "customer_name": None,
    "email": None,
    "source_file": "test_batch.json"
}

print(process_event(delete_test, run_id_2))

ACCEPTED


In [38]:
insert_test = {
    "event_id": "E203",
    "customer_id": "C201",
    "event_type": "INSERT",
    "event_version": 3,
    "event_timestamp": "2026-11-13T14:30:00Z",
    "customer_name": "Arun",
    "email": "arun@example.com",
    "source_file": "test_batch.json"
}

print(process_event(insert_test, run_id_2))

ACCEPTED


In [39]:
late_test = {
    "event_id": "E204",
    "customer_id": "C201",
    "event_type": "UPDATE",
    "event_version": 2,
    "event_timestamp": "2026-11-13T14:15:00Z",
    "customer_name": "Arun Old",
    "email": "old@example.com",
    "source_file": "late_batch.json"
}

print(process_event(late_test, run_id_2))

ACCEPTED


In [40]:
invalid_test = {
    "event_id": "E205",
    "customer_id": "C202",
    "event_type": "INSERT",
    "event_version": 0,
    "event_timestamp": "wrong-date",
    "customer_name": "",
    "email": "invalid-email",
    "source_file": "bad_batch.json"
}

print(process_event(invalid_test, run_id_2))

REJECTED


In [41]:
print(process_event(test_event, run_id_2))

DUPLICATE


In [42]:
def process_batch(events, run_id):
    results = []

    print(f"Starting batch processing. Total events: {len(events)}")

    try:
        for event in events:
            result = process_event(event, run_id)

            results.append({
                "event_id": event.get("event_id"),
                "customer_id": event.get("customer_id"),
                "status": result
            })

        print("\nBatch processing completed!")

    except Exception as error:
        print("\nBatch processing failed:", error)

        connection = get_connection()
        cursor = connection.cursor()

        cursor.execute("""
            UPDATE pipeline_runs
            SET status = 'FAILED',
                end_time = %s,
                error_message = %s
            WHERE run_id = %s
        """, (datetime.now(), str(error), run_id))

        connection.commit()
        cursor.close()
        connection.close()

        raise

    return results

In [43]:
batch_events = [
    {
        "event_id": "B001",
        "customer_id": "C300",
        "event_type": "INSERT",
        "event_version": 1,
        "event_timestamp": "2026-11-13T10:00:00",
        "customer_name": "Ramesh",
        "email": "ramesh@example.com"
    },
    {
        "event_id": "B002",
        "customer_id": "C300",
        "event_type": "UPDATE",
        "event_version": 2,
        "event_timestamp": "2026-11-13T11:00:00",
        "customer_name": "Ramesh Kumar",
        "email": "rameshk@example.com"
    }
]

batch_run_id = start_pipeline_run()

batch_results = process_batch(batch_events, batch_run_id)

print(batch_results)

Starting batch processing. Total events: 2

Batch processing completed!
[{'event_id': 'B001', 'customer_id': 'C300', 'status': 'ACCEPTED'}, {'event_id': 'B002', 'customer_id': 'C300', 'status': 'ACCEPTED'}]


In [44]:
# Test 1: Submit B001 again (duplicate)
duplicate_result = process_event(batch_events[0], batch_run_id)
print("Duplicate test:", duplicate_result)

# Test 2: Submit an invalid event
invalid_event = {
    "event_id": "B003",
    "customer_id": "C301",
    "event_type": "INSERT",
    "event_version": 0,
    "event_timestamp": "wrong-date",
    "customer_name": "",
    "email": "invalid-email"
}

invalid_result = process_event(invalid_event, batch_run_id)
print("Invalid test:", invalid_result)

Duplicate test: DUPLICATE
Invalid test: REJECTED


In [46]:
empty_run_id = start_pipeline_run()

empty_results = process_batch([], empty_run_id)

print("Empty batch result:", empty_results)

Starting batch processing. Total events: 0

Batch processing completed!
Empty batch result: []


In [47]:
def complete_pipeline_run(run_id):
    connection = get_connection()
    cursor = connection.cursor()

    try:
        cursor.execute("""
            UPDATE pipeline_runs
            SET status = 'COMPLETED',
                end_time = %s
            WHERE run_id = %s
              AND status = 'RUNNING'
        """, (datetime.now(), run_id))

        connection.commit()
        print(f"Run {run_id} marked as COMPLETED.")

    except Exception:
        connection.rollback()
        raise

    finally:
        cursor.close()
        connection.close()

In [48]:
complete_pipeline_run(batch_run_id)
complete_pipeline_run(empty_run_id)

Run 3 marked as COMPLETED.
Run 5 marked as COMPLETED.


In [49]:
complete_pipeline_run(2)
complete_pipeline_run(4)

Run 2 marked as COMPLETED.
Run 4 marked as COMPLETED.


In [53]:
%mkdir tests

In [57]:
!python -m pytest tests/test_pipeline.py -v

============================= test session starts =============================
platform win32 -- Python 3.13.9, pytest-8.4.2, pluggy-1.5.0 -- C:\ProgramData\anaconda3\python.exe
cachedir: .pytest_cache
rootdir: C:\Users\Welcome
plugins: anyio-4.10.0
collecting ... collected 3 items

tests/test_pipeline.py::test_valid_event PASSED                          [ 33%]
tests/test_pipeline.py::test_missing_customer_name PASSED                [ 66%]
tests/test_pipeline.py::test_invalid_event_version PASSED                [100%]

============================== 3 passed in 0.09s ==============================
